In [3]:
import scipy.io
import numpy as np
import matplotlib.pyplot as plt
import os
import matplotlib
import pandas as pd
import cv2
import logging
import json
from matplotlib.path import Path
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch
from datetime import datetime

# Set Matplotlib to non-interactive mode for batch processing
matplotlib.use('Agg')

# Configure Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)

# ====================================================================
# 1. CONFIGURATION PARAMETERS
# ====================================================================
BINDING_RADIUS_PX = 0.8480
BINDING_MIN_FRAMES = 5
PIXEL_SIZE_NM = 117

# Minimum fraction of a track's points that must remain inside the cell
MIN_INSIDE_RATIO = 1.0

COLOR_STATIC = '#e69f00'    
COLOR_TRANSIENT = '#00b4f0' 
COLOR_MOBILE = '#3877cb'    
ALPHA_PALETTE = 0.85        

oufti_dir = r"oufti_dir"
utrack_root_dir = r"utrack_dir"
output_dir = r"output_dir"
excel_output_path = os.path.join(output_dir, 'SMT_State_Analysis_Test.xlsx')

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# ====================================================================
# 2. HELPER FUNCTIONS
# ====================================================================
def recursive_mesh_extractor(obj, mesh_list):
    """Recursively finds Nx4 mesh matrices within the MATLAB structure."""
    if not isinstance(obj, (np.ndarray, np.void)): return
    if isinstance(obj, np.ndarray):
        if obj.ndim == 2 and obj.shape[1] == 4 and obj.shape[0] > 5:
            if np.issubdtype(obj.dtype, np.number):
                mesh_list.append(obj); return 
    if hasattr(obj, 'dtype') and obj.dtype.names is not None:
        for name in obj.dtype.names:
            recursive_mesh_extractor(obj[name], mesh_list)
    elif isinstance(obj, np.ndarray):
        for item in obj.flatten():
            recursive_mesh_extractor(item, mesh_list)

def find_binding_events(x, y, radius, min_frames):
    """Identifies all time points (indices) within a track that belong to a bound state."""
    if len(x) < min_frames: return []
    bound_indices = []
    for start_idx in range(len(x) - min_frames + 1):
        window_x = x[start_idx : start_idx + min_frames]
        window_y = y[start_idx : start_idx + min_frames]
        displacements = np.sqrt((window_x - window_x[0])**2 + (window_y - window_y[0])**2)
        if np.max(displacements) < radius:
            bound_indices.extend(range(start_idx, start_idx + min_frames))
    return sorted(list(set(bound_indices)))

def group_contiguous_indices(indices):
    """Groups a flat list of indices into distinct, contiguous events."""
    if not indices: return []
    events = []
    current_event = [indices[0]]
    for i in range(1, len(indices)):
        if indices[i] == indices[i-1] + 1:
            current_event.append(indices[i])
        else:
            events.append(current_event)
            current_event = [indices[i]]
    events.append(current_event)
    return events

def classify_track_state(bound_indices, total_frames):
    """Classifies a track based on binding events: 'Static', 'Transient', or 'Mobile'."""
    if not bound_indices: return 'Mobile'
    bound_mask = np.zeros(total_frames, dtype=bool)
    bound_mask[bound_indices] = True
    return 'Static' if np.all(bound_mask) else 'Transient'

def load_tracks_from_utrack(path_utrack):
    """Loads and processes raw tracks from the u-track MATLAB file."""
    mat_utrack = scipy.io.loadmat(path_utrack)
    raw_tracks = np.atleast_1d(mat_utrack['tracksFinal']).flatten()
    processed_trajectories = []
    
    for t in raw_tracks:
        try:
            m = t['tracksCoordAmpCG']
            while isinstance(m, np.ndarray) and m.ndim > 2: m = m[0]
            m_res = m.flatten().reshape(-1, 8) 
            x, y = m_res[:, 0], m_res[:, 1]
            
            valid_mask = ~np.isnan(x)
            if sum(valid_mask) >= BINDING_MIN_FRAMES:
                track_x, track_y = x[valid_mask], y[valid_mask]
                bound_indices = find_binding_events(track_x, track_y, BINDING_RADIUS_PX, BINDING_MIN_FRAMES)
                
                processed_trajectories.append({
                    'x': track_x, 'y': track_y, 
                    'bound_indices': bound_indices,
                    'state_type': classify_track_state(bound_indices, len(track_x))
                })
        except Exception:
            continue
    return processed_trajectories

def get_tracks_in_cell_vectorized(cell_contour, all_tracks, min_inside_ratio=0.1):
    """Vectorized check to determine which tracks belong to a cell."""
    poly_path = Path(cell_contour)
    filtered_tracks = []
    stats = {'total': len(all_tracks), 'passed': 0, 'failed_start': 0, 'failed_ratio': 0}
    
    for tr in all_tracks:
        if not poly_path.contains_point((tr['x'][0], tr['y'][0])):
            stats['failed_start'] += 1
            continue
        
        if min_inside_ratio > 0:
            track_points = np.column_stack([tr['x'], tr['y']])
            inside_mask = poly_path.contains_points(track_points)
            inside_ratio = np.sum(inside_mask) / len(inside_mask)
            
            if inside_ratio < min_inside_ratio:
                stats['failed_ratio'] += 1
                continue
        
        filtered_tracks.append(tr)
        stats['passed'] += 1
    
    return filtered_tracks, stats

def analyze_cell_tracks(cell_contour, tracks_in_cell, series_id, cell_id):
    """Calculates distance maps and extracts metrics for all tracks inside a specific cell."""
    x_min, y_min = np.floor(np.min(cell_contour, axis=0)).astype(int) - 5
    x_max, y_max = np.ceil(np.max(cell_contour, axis=0)).astype(int) + 5
    mask_img = np.zeros((y_max - y_min, x_max - x_min), dtype=np.uint8)
    cv2.fillPoly(mask_img, [(cell_contour - [x_min, y_min]).astype(np.int32)], 255)
    dist_map = cv2.distanceTransform(mask_img, cv2.DIST_L2, 5)

    max_dist_px = np.max(dist_map)
    if max_dist_px == 0: max_dist_px = 1 

    n_static, n_transient, n_mobile = 0, 0, 0
    bound_distances_nm = []
    individual_data = []
    
    for tr in tracks_in_cell:
        state = tr['state_type']
        if state == 'Static': n_static += 1
        elif state == 'Transient': n_transient += 1
        else: n_mobile += 1

        events = group_contiguous_indices(tr['bound_indices'])
        
        if not events:
            ref_x, ref_y = tr['x'][0], tr['y'][0]
            py = int(np.clip(round(ref_y - y_min), 0, dist_map.shape[0] - 1))
            px = int(np.clip(round(ref_x - x_min), 0, dist_map.shape[1] - 1))
            
            dist_px = dist_map[py, px]
            dist_nm = dist_px * PIXEL_SIZE_NM
            normalized_dist_255 = (dist_px / max_dist_px) * 255.0

            individual_data.append({
                'Series': series_id, 'Cell_ID': cell_id, 'Event_ID': 1,
                'Type': state, 
                'Dist_to_Membrane_nm': round(dist_nm, 1),
                'Relative_Position_255': round(normalized_dist_255, 1),
                'Duration_Frames': len(tr['x'])
            })
        else:
            for event_idx, event in enumerate(events):
                ref_idx = event[0]
                ref_x, ref_y = tr['x'][ref_idx], tr['y'][ref_idx]
                
                py = int(np.clip(round(ref_y - y_min), 0, dist_map.shape[0] - 1))
                px = int(np.clip(round(ref_x - x_min), 0, dist_map.shape[1] - 1))
                
                dist_px = dist_map[py, px]
                dist_nm = dist_px * PIXEL_SIZE_NM
                normalized_dist_255 = (dist_px / max_dist_px) * 255.0

                bound_distances_nm.append(dist_nm)
                individual_data.append({
                    'Series': series_id, 'Cell_ID': cell_id, 'Event_ID': event_idx + 1,
                    'Type': state, 
                    'Dist_to_Membrane_nm': round(dist_nm, 1),
                    'Relative_Position_255': round(normalized_dist_255, 1),
                    'Duration_Frames': len(tr['x'])
                })

    area_px2 = cv2.contourArea(cell_contour.astype(np.float32))
    area_um2 = (area_px2 * (PIXEL_SIZE_NM**2)) / 1e6

    cell_summary = {
        'Series': series_id, 'Cell_ID': cell_id,
        'Area_um2': round(area_um2, 3),
        'Total_Tracks': len(tracks_in_cell),
        'N_Static': n_static, 'N_Transient': n_transient, 'N_Mobile': n_mobile,
        'Total_Binding_Events': len(bound_distances_nm),
        'Mean_Binding_Dist_nm': round(np.mean(bound_distances_nm), 1) if bound_distances_nm else 0,
        'Mean_Relative_Position_255': round(np.mean([d['Relative_Position_255'] for d in individual_data]), 1) if individual_data else 0 
    }
    
    return cell_summary, individual_data

def plot_fov(cell_meshes, all_tracks, series_id, output_dir):
    """Generates the publication-ready plot using LineCollection for speed."""
    fig, ax = plt.subplots(figsize=(10, 8))
    
    for i, mesh in enumerate(cell_meshes):
        ax.plot(mesh[:, 0], mesh[:, 1], color='gray', alpha=0.3)
        ax.plot(mesh[:, 2], mesh[:, 3], color='gray', alpha=0.3)
        ax.text(mesh[0,0], mesh[0,1]-2, f"Cell {i+1}", color='white', fontsize=7, 
                bbox=dict(facecolor='black', alpha=0.5))

    mobile_segs = [np.column_stack([tr['x'], tr['y']]) for tr in all_tracks if tr['state_type'] == 'Mobile']
    static_segs = [np.column_stack([tr['x'], tr['y']]) for tr in all_tracks if tr['state_type'] == 'Static']
    transient_segs = [np.column_stack([tr['x'], tr['y']]) for tr in all_tracks if tr['state_type'] == 'Transient']

    if mobile_segs:
        ax.add_collection(LineCollection(mobile_segs, colors=COLOR_MOBILE, alpha=ALPHA_PALETTE, linewidths=0.5))
    if static_segs:
        ax.add_collection(LineCollection(static_segs, colors=COLOR_STATIC, alpha=ALPHA_PALETTE, linewidths=1.2))
    if transient_segs:
        ax.add_collection(LineCollection(transient_segs, colors=COLOR_TRANSIENT, alpha=ALPHA_PALETTE, linewidths=0.8, linestyles='--'))

    for tr in all_tracks:
        if tr['state_type'] == 'Static':
            ax.scatter(tr['x'][0], tr['y'][0], color='yellow', s=6, zorder=11)
        elif tr['state_type'] == 'Transient':
            events = group_contiguous_indices(tr['bound_indices'])
            for event in events:
                idx = event[0]
                ax.scatter(tr['x'][idx], tr['y'][idx], color='yellow', s=4, zorder=6)

    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.set_title(f"3-State SMT: {series_id} (Orange: Static, Dashed Cyan: Transient, Blue: Mobile)")
    plt.savefig(os.path.join(output_dir, f"full_analysis_{series_id}.png"), dpi=600)
    plt.close(fig)

# ====================================================================
# 3. NORMALIZED OVERLAY PLOTTING FUNCTIONS (ENHANCED)
# ====================================================================
def plot_normalized_tracks(all_cells_data, output_dir, plot_title, target_width=100, target_height=40):
    """
    Creates ONE single overlay plot of ALL tracks from ALL provided cells.
    Features: SVG (vector) & PNG (rasterized) export, scale bar, split transient tracks.
    """
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # 1. Draw the average cell boundary using FancyBboxPatch (perfect stadium)
    stadium_patch = FancyBboxPatch(
        (-target_width/2, -target_height/2), 
        target_width, 
        target_height,
        boxstyle=f"round,pad=0,rounding_size={target_height/2}",
        linewidth=2.5,
        edgecolor='black',
        facecolor='lightgray',
        alpha=0.4,
        zorder=1
    )
    ax.add_patch(stadium_patch)
    
    # Lists to hold segments
    mobile_segs = []
    static_segs = []
    transient_static_segs = []
    transient_mobile_segs = []
    
    # Calculate average physical dimensions for the scale bar
    widths_px = []
    for cell_contour, _ in all_cells_data:
        x_min, x_max = np.min(cell_contour[:, 0]), np.max(cell_contour[:, 0])
        widths_px.append(x_max - x_min)
    
    avg_width_px = np.mean(widths_px) if widths_px else 1
    avg_width_nm = avg_width_px * PIXEL_SIZE_NM
    scale_bar_length_norm = (1000 / avg_width_nm) * target_width
    
    # 2. Loop through ALL cells and normalize their tracks
    for cell_contour, tracks in all_cells_data:
        # Step A: Find the cell center
        pts = cell_contour.astype(np.float64)
        cx, cy = np.mean(pts[:, 0]), np.mean(pts[:, 1])
        
        # Step B: Find orientation using Principal Component Analysis (PCA)
        cov_mat = np.cov((pts - [cx, cy]).T)
        evals, evecs = np.linalg.eig(cov_mat)
        major_axis = evecs[:, np.argmax(evals)]
        angle = np.arctan2(major_axis[1], major_axis[0])
        
        # Step C: Define a function to rotate points to the X-axis
        def align_and_center(x, y):
            sx = x - cx
            sy = y - cy
            # Apply 2D Rotation Matrix by -angle
            rot_x = sx * np.cos(-angle) - sy * np.sin(-angle)
            rot_y = sx * np.sin(-angle) + sy * np.cos(-angle)
            return rot_x, rot_y
            
        # Step D: Rotate the contour to calculate the TRUE width and height
        rot_contour_x, rot_contour_y = align_and_center(pts[:, 0], pts[:, 1])
        
        x_min, x_max = np.min(rot_contour_x), np.max(rot_contour_x)
        y_min, y_max = np.min(rot_contour_y), np.max(rot_contour_y)
        
        cell_width = (x_max - x_min) if (x_max - x_min) > 0 else 1
        cell_height = (y_max - y_min) if (y_max - y_min) > 0 else 1
        
        # Step E: Update the normalization function
        def norm_coords(x, y):
            rx, ry = align_and_center(x, y)
            nx = (rx / cell_width) * target_width
            ny = (ry / cell_height) * target_height
            return np.column_stack([nx, ny])
        
        # Step F: Process the tracks using the updated normalization
        for tr in tracks:
            if tr['state_type'] == 'Mobile':
                mobile_segs.append(norm_coords(tr['x'], tr['y']))
            elif tr['state_type'] == 'Static':
                static_segs.append(norm_coords(tr['x'], tr['y']))
            elif tr['state_type'] == 'Transient':
                n_frames = len(tr['x'])
                is_bound = np.zeros(n_frames, dtype=bool)
                if tr['bound_indices']:
                    is_bound[tr['bound_indices']] = True
                
                start = 0
                while start < n_frames:
                    end = start + 1
                    while end < n_frames and is_bound[end] == is_bound[start]:
                        end += 1
                    
                    if end - start == 1:
                        seg_x = np.array([tr['x'][start], tr['x'][start]])
                        seg_y = np.array([tr['y'][start], tr['y'][start]])
                    else:
                        seg_x = tr['x'][start:end]
                        seg_y = tr['y'][start:end]
                    
                    seg_coords = norm_coords(seg_x, seg_y)
                    
                    if is_bound[start]:
                        transient_static_segs.append(seg_coords)
                    else:
                        transient_mobile_segs.append(seg_coords)
                    
                    start = end

    # 3. Create LineCollection objects
    lc_mobile = LineCollection(mobile_segs, colors=COLOR_MOBILE, alpha=0.25, linewidths=0.4, zorder=2)
    lc_static = LineCollection(static_segs, colors=COLOR_STATIC, alpha=0.7, linewidths=1.5, zorder=3)
    lc_trans_static = LineCollection(transient_static_segs, colors=COLOR_STATIC, alpha=0.8, linewidths=2.0, zorder=3)
    lc_trans_mobile = LineCollection(transient_mobile_segs, colors=COLOR_TRANSIENT, alpha=0.4, linewidths=1.2, linestyles='--', zorder=2)

    ax.add_collection(lc_mobile)
    ax.add_collection(lc_static)
    ax.add_collection(lc_trans_static)
    ax.add_collection(lc_trans_mobile)

    # 4. Calculate stats
    total_mobile = len(mobile_segs)
    total_static = len(static_segs)
    total_transient = len(transient_static_segs) + len(transient_mobile_segs)
    total_tracks = total_mobile + total_static + total_transient
    
    if total_tracks > 0:
        pct_mobile = (total_mobile / total_tracks) * 100
        pct_static = (total_static / total_tracks) * 100
        pct_transient = (total_transient / total_tracks) * 100
    else:
        pct_mobile = pct_static = pct_transient = 0

    # 5. Add Legend
    legend_elements = [
        Line2D([0], [0], color=COLOR_MOBILE, lw=2.5, alpha=0.7, label=f'Mobile: {pct_mobile:.1f}% ({total_mobile})'),
        Line2D([0], [0], color=COLOR_STATIC, lw=2.5, alpha=0.9, label=f'Static: {pct_static:.1f}% ({total_static})'),
        Line2D([0], [0], color=COLOR_TRANSIENT, lw=2.5, alpha=0.7, linestyle='--', label=f'Transient: {pct_transient:.1f}% ({total_transient})')
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=11, framealpha=0.95, edgecolor='black')

    # 6. Add Scale Bar
    scale_bar_y = -target_height/2 - 8
    scale_bar_x_start = -target_width/2 + 10
    ax.plot([scale_bar_x_start, scale_bar_x_start + scale_bar_length_norm], 
            [scale_bar_y, scale_bar_y], 'k-', linewidth=3, zorder=5)
    ax.text(scale_bar_x_start + scale_bar_length_norm/2, scale_bar_y - 3, '1 µm', 
            ha='center', va='top', fontsize=10, fontweight='bold', zorder=5)

    # 7. Formatting
    ax.set_aspect('equal')
    ax.set_xlim(-target_width/2 - 15, target_width/2 + 15)
    ax.set_ylim(-target_height/2 - 15, target_height/2 + 15)
    ax.axis('off')
    
    ax.set_title(f"{plot_title}\n({len(all_cells_data)} cells, {total_tracks} total tracks)", 
                 pad=20, fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    
    safe_title = plot_title.replace(" ", "_").replace("/", "_")
    png_path = os.path.join(output_dir, f"normalized_overlay_{safe_title}.png")
    svg_path = os.path.join(output_dir, f"normalized_overlay_{safe_title}.svg")

    # 8. Save PNG (Rasterized for performance and smaller file size)
    for lc in [lc_mobile, lc_static, lc_trans_static, lc_trans_mobile]:
        lc.set_rasterized(True)
    plt.savefig(png_path, dpi=600, bbox_inches='tight')
    
    # 9. Save SVG (Pure Vector for infinite scaling and perfect quality)
    for lc in [lc_mobile, lc_static, lc_trans_static, lc_trans_mobile]:
        lc.set_rasterized(False)
    plt.savefig(svg_path, bbox_inches='tight', format='svg')
    
    plt.close(fig)
    
    logging.info(f"  -> Saved final combined overlay to: {png_path} and {svg_path}")
    
    return {'total': total_tracks, 'mobile': total_mobile, 'static': total_static, 'transient': total_transient}

# ====================================================================
# 4. MAIN EXECUTION LOOP
# ====================================================================
oufti_files = [f for f in os.listdir(oufti_dir) if f.endswith('.mat')]
cell_summary_data = []
individual_track_data = []

# Initialize global list to collect data from ALL FOVs for one single combined plot
all_normalized_data_global = []

logging.info(f"--- Starting analysis (Categorization: 3-States, Scale: {PIXEL_SIZE_NM} nm/px) ---")
if MIN_INSIDE_RATIO > 0:
    logging.info(f"--- Track filtering: Minimum {MIN_INSIDE_RATIO*100}% of points must remain inside cell ---")
else:
    logging.info(f"--- Track filtering: DISABLED (all tracks starting inside cell will be analyzed) ---")

for filename in oufti_files:
    series_id = os.path.splitext(filename)[0]
    path_oufti = os.path.join(oufti_dir, filename)
    path_utrack = os.path.join(utrack_root_dir, series_id, 'TrackingPackage', 'tracks', 'Channel_1_tracking_result.mat')

    if not os.path.exists(path_utrack):
        logging.warning(f"️ {series_id}: u-track file not found, skipping.")
        continue

    try:
        mat_oufti = scipy.io.loadmat(path_oufti)
        cell_meshes = []
        if 'cellListN' in mat_oufti: recursive_mesh_extractor(mat_oufti['cellListN'], cell_meshes)
        if not cell_meshes and 'cellList' in mat_oufti: recursive_mesh_extractor(mat_oufti['cellList'], cell_meshes)
        
        all_tracks = load_tracks_from_utrack(path_utrack)
        logging.info(f"📊 {series_id}: Loaded {len(all_tracks)} tracks, {len(cell_meshes)} cells")

        for i, mesh in enumerate(cell_meshes):
            contour = np.vstack((mesh[:, 0:2], mesh[::-1, 2:4]))
            tracks_in_cell, stats = get_tracks_in_cell_vectorized(contour, all_tracks, MIN_INSIDE_RATIO)
            
            if stats['passed'] == 0:
                logging.debug(f"  Cell {i+1}: 0 tracks passed filtering")
            
            # Analyze and record the cell
            summary, ind_data = analyze_cell_tracks(contour, tracks_in_cell, series_id, i + 1)
            cell_summary_data.append(summary)
            individual_track_data.extend(ind_data)
            
            # Append to global list for the final combined plot
            if tracks_in_cell:
                all_normalized_data_global.append((contour, tracks_in_cell))
        
        # Generate standard FOV plot (one per series)
        plot_fov(cell_meshes, all_tracks, series_id, output_dir)
        logging.info(f"✅ {series_id}: Processed successfully.\n")

    except Exception as e: 
        logging.error(f"❌ {series_id}: Error -> {e}")

# ====================================================================
# 5. GENERATE FINAL COMBINED OVERLAY & EXPORT
# ====================================================================
if all_normalized_data_global:
    logging.info(f"📈 Generating SINGLE combined normalized overlay for {len(all_normalized_data_global)} total cells...")
    stats = plot_normalized_tracks(all_normalized_data_global, output_dir, "All_Cells_Combined")
    logging.info(f"   -> Final Stats: {stats['total']} tracks (M:{stats['mobile']}, S:{stats['static']}, T:{stats['transient']})")
else:
    logging.warning("No tracks passed filtering across all FOVs. Skipping combined overlay.")

if individual_track_data or cell_summary_data:
    with pd.ExcelWriter(excel_output_path) as writer:
        pd.DataFrame(cell_summary_data).to_excel(writer, sheet_name='Cell_Summary', index=False)
        pd.DataFrame(individual_track_data).to_excel(writer, sheet_name='Individual_Tracks', index=False)
    logging.info(f"--- SUCCESS ---")
    logging.info(f"Excel report generated: {excel_output_path}")
    logging.info(f"Total cells analyzed: {len(cell_summary_data)}")
    logging.info(f"Total track events recorded: {len(individual_track_data)}")
    
    # --- NEW AUTO-METADATA BLOCK ---
    metadata = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "BINDING_RADIUS_PX": BINDING_RADIUS_PX,
        "BINDING_MIN_FRAMES": BINDING_MIN_FRAMES,
        "PIXEL_SIZE_NM": PIXEL_SIZE_NM,
        "MIN_INSIDE_RATIO": MIN_INSIDE_RATIO,
        "Input_Oufti_Dir": oufti_dir,
        "Input_uTrack_Dir": utrack_root_dir,
        "Total_Cells_Analyzed": len(cell_summary_data),
        "Total_Tracks_Filtered": stats['total'] if 'stats' in locals() else 0
    }
    
    metadata_path = os.path.join(output_dir, 'analysis_parameters.json')
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=4)
    logging.info(f"Metadata log saved: {metadata_path}")

else:
    logging.warning("No data was processed. Excel file was not created.")


2026-09-01 14:45:32,034 - INFO - --- Starting analysis (Categorization: 3-States, Scale: 117 nm/px) ---
2026-09-01 14:45:32,034 - INFO - --- Track filtering: Minimum 100.0% of points must remain inside cell ---
2026-09-01 14:45:32,052 - INFO - 📊 series091: Loaded 7 tracks, 1 cells
2026-09-01 14:45:33,181 - INFO - ✅ series091: Processed successfully.

2026-09-01 14:45:33,192 - INFO - 📊 series021: Loaded 11 tracks, 1 cells
2026-09-01 14:45:33,947 - INFO - ✅ series021: Processed successfully.

2026-09-01 14:45:33,974 - INFO - 📊 series031: Loaded 2 tracks, 2 cells
2026-09-01 14:45:34,781 - INFO - ✅ series031: Processed successfully.

2026-09-01 14:45:34,796 - INFO - 📊 series041: Loaded 5 tracks, 2 cells
2026-09-01 14:45:35,582 - INFO - ✅ series041: Processed successfully.

2026-09-01 14:45:35,599 - INFO - 📊 series042: Loaded 10 tracks, 1 cells
2026-09-01 14:45:36,468 - INFO - ✅ series042: Processed successfully.

2026-09-01 14:45:36,485 - INFO - 📊 series044: Loaded 34 tracks, 1 cells
2026-